In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_4.py ──
"""
Shared infrastructure for MLFP03 Exercise 4 — Gradient Boosting Deep Dive.

Contains: Singapore credit-scoring data loader, preprocessing pipeline
wrapper, numpy/polars conversion, model-zoo factories (XGBoost/LightGBM/
CatBoost with identical defaults), evaluation metric helpers, and the
output directory used by every technique file.

Technique-specific code (from-scratch boosting loops, split-gain formulas,
hyperparameter sweeps, early-stopping analysis) does NOT belong here — it
lives in the per-technique files under `modules/mlfp03/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONFIG — output directory, random seeds, dataset tag
# ════════════════════════════════════════════════════════════════════════

SEED = 42
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"

OUTPUT_DIR = Path("outputs") / "mlfp03" / "ex_4_boosting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit-scoring, shared across all four techniques
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> pl.DataFrame:
    """Load the Singapore credit-scoring dataset via MLFPDataLoader."""
    loader = MLFPDataLoader()
    return loader.load(DATASET_MODULE, DATASET_FILE)


def prepare_credit_split() -> dict[str, Any]:
    """Load credit data and return a train/test split ready for boosting.

    Returns a dict with:
      X_train, y_train, X_test, y_test : numpy arrays
      feature_names                    : list[str]
      default_rate                     : float (positive-class prevalence)

    Tree models do not need normalisation, so we set ``normalize=False``.
    Categoricals are ordinal-encoded because XGBoost/LightGBM expect numeric
    input; CatBoost would accept raw categoricals but we keep the pipeline
    consistent across all three libraries for a fair comparison.
    """
    credit = load_credit_data()

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=credit,
        target=TARGET_COLUMN,
        train_size=0.8,
        seed=SEED,
        normalize=False,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(credit[TARGET_COLUMN].mean()),
    }


# ════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES — identical defaults for fair comparison
# ════════════════════════════════════════════════════════════════════════


def make_xgboost(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build an XGBoost classifier with course-standard defaults."""
    import xgboost as xgb

    return xgb.XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        eval_metric="logloss",
        random_state=SEED,
        verbosity=0,
        **extra,
    )


def make_lightgbm(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build a LightGBM classifier with course-standard defaults."""
    import lightgbm as lgb

    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=31,
        random_state=SEED,
        verbose=-1,
        **extra,
    )


def make_catboost(
    iterations: int = 500,
    learning_rate: float = 0.1,
    depth: int = 6,
    **extra: Any,
):
    """Build a CatBoost classifier with course-standard defaults."""
    import catboost as cb

    return cb.CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        random_seed=SEED,
        verbose=0,
        **extra,
    )


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — shared metric helpers for imbalanced binary classification
# ════════════════════════════════════════════════════════════════════════


def evaluate_classifier(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Return the full boosting-eval metric bundle.

    AUC-PR is the primary metric — with a 12% default rate, AUC-ROC rewards
    models that rank negatives correctly even when they miss every default.
    """
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred)),
    }


def print_metrics(
    name: str, metrics: dict[str, float], train_time: float | None = None
) -> None:
    """One-line headline: AUC-ROC | AUC-PR | Log Loss | F1 | Time."""
    time_str = f" | time={train_time:.2f}s" if train_time is not None else ""
    print(
        f"  {name}: "
        f"AUC-ROC={metrics['auc_roc']:.4f} | "
        f"AUC-PR={metrics['auc_pr']:.4f} | "
        f"log_loss={metrics['log_loss']:.4f} | "
        f"F1={metrics['f1']:.4f}"
        f"{time_str}"
    )


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH GRADIENT BOOSTING (used by 01_boosting_theory.py)
# ════════════════════════════════════════════════════════════════════════


def xgb_split_gain(
    g_left: float,
    h_left: float,
    g_right: float,
    h_right: float,
    lambda_reg: float = 1.0,
    gamma: float = 0.0,
) -> float:
    """XGBoost split-gain formula from 2nd-order Taylor expansion of log-loss.

    Gain = ½ [G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ)] - γ
    """
    left_score = g_left**2 / (h_left + lambda_reg)
    right_score = g_right**2 / (h_right + lambda_reg)
    parent_score = (g_left + g_right) ** 2 / (h_left + h_right + lambda_reg)
    return 0.5 * (left_score + right_score - parent_score) - gamma


def make_1d_demo(n: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Generate a 1D logistic-shaped binary classification demo.

    Used by the from-scratch boosting loop to show that residual-fitting
    converges in just a handful of rounds.
    """
    rng = np.random.default_rng(SEED)
    x = rng.uniform(0, 1, n).reshape(-1, 1)
    true_proba = 1 / (1 + np.exp(-10 * (x.ravel() - 0.5)))
    y = rng.binomial(1, true_proba)
    return x, y




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 4.2: XGBoost on Singapore Credit Data
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Train an XGBoost classifier on a real imbalanced credit dataset
#   - Connect XGBoost hyperparameters back to the theory in 4.1
#   - Use AUC-PR as the primary metric for 12%-positive data
#   - Extract and rank gain-based feature importances
#   - Explain why XGBoost is the default choice for tabular data
#
# PREREQUISITES: Exercise 4.1 (boosting theory, split-gain formula).
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — hyperparameters as theory dials
#   2. Build — XGBoost with course-standard defaults
#   3. Train — fit, time the training, evaluate
#   4. Visualise — feature importance bar chart
#   5. Apply — DBS credit risk team explainability for MAS FEAT
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import plotly.graph_objects as go
from dotenv import load_dotenv


load_dotenv()



## THEORY — XGBoost hyperparameters as theory dials


In [ ]:
# learning_rate (η)  → additive-model step size
# max_depth          → tree size / interaction order
# n_estimators       → number of boosting rounds
# reg_lambda (λ)     → L2 on leaf weights (from split-gain formula)
# gamma (γ)          → minimum gain to accept a split
# subsample          → stochastic row sampling per tree
# colsample_bytree   → stochastic column sampling per tree



## TASK 2 — BUILD the XGBoost classifier


In [ ]:
print("\n" + "=" * 70)
print("  XGBoost on Singapore Credit Scoring")
print("=" * 70)

data = prepare_credit_split()
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
feature_names = data["feature_names"]

print(f"\n  Train: {X_train.shape} | Test: {X_test.shape}")
print(f"  Features: {len(feature_names)}")
print(f"  Default rate: {data['default_rate']:.2%}")

# TODO: Build an XGBoost classifier via make_xgboost with:
# n_estimators=500, learning_rate=0.1, max_depth=6
model = ____



## TASK 3 — TRAIN on Singapore credit data


In [ ]:
print("\n  Training XGBoost (500 rounds, η=0.1, depth=6)...")
t0 = time.perf_counter()

# TODO: Fit the model using X_train, y_train.
# Pass eval_set=[(X_test, y_test)] and verbose=False.
____

train_time = time.perf_counter() - t0

# TODO: Predict class-1 probabilities for X_test.
# Hint: model.predict_proba returns an (n, 2) array — take column [:, 1].
y_proba = ____

# TODO: Call evaluate_classifier(y_test, y_proba) to get the metric bundle.
metrics = ____

print_metrics("XGBoost", metrics, train_time=train_time)



### Checkpoint 1


In [ ]:
assert metrics["auc_roc"] > 0.7, "XGBoost should beat 0.7 AUC-ROC on credit data"
assert metrics["auc_pr"] > 0.3, "XGBoost AUC-PR should clear 0.3 (12% base rate)"
print("\n[ok] Checkpoint 1 passed — XGBoost trained and evaluated\n")



## TASK 4 — VISUALISE feature importance


In [ ]:
# XGBoost uses gain-based importance by default: the total reduction in
# loss attributed to each feature via its split-gain contributions.

# TODO: Pull the importance vector from the trained model.
# Hint: model.feature_importances_
importances = ____

# TODO: Sort zip(feature_names, importances) by importance descending,
# take top 15.
ranked = ____
top_15 = ranked[:15]

print("  --- Top-15 Features by XGBoost Gain Importance ---")
print(f"  {'Rank':>4}  {'Feature':<30}  {'Gain':>10}")
print("  " + "─" * 50)
for rank, (name, importance) in enumerate(top_15, start=1):
    print(f"  {rank:>4}  {name:<30}  {importance:>10.4f}")

names = [name for name, _ in top_15][::-1]
values = [float(v) for _, v in top_15][::-1]

fig = go.Figure(
    go.Bar(
        x=values,
        y=names,
        orientation="h",
        marker=dict(color=values, colorscale="Blues"),
    )
)
fig.update_layout(
    title="XGBoost Feature Importance — Singapore Credit Default",
    xaxis_title="Gain",
    height=520,
)
viz_path = OUTPUT_DIR / "ex4_02_xgboost_feature_importance.html"
fig.write_html(viz_path)
print(f"\n  Saved: {viz_path}")



### Checkpoint 2


In [ ]:
assert len(importances) == len(feature_names)
assert sum(importances) > 0
print("\n[ok] Checkpoint 2 passed — feature importance extracted and visualised\n")



## TASK 5 — APPLY: DBS Credit Risk Team Explainability


In [ ]:
# SCENARIO: DBS must explain every automated decline to MAS under the
# FEAT principles (Fairness, Ethics, Accountability, Transparency). The
# XGBoost gain importance above is the first artifact the risk team
# delivers. Expected top features: Debt Service Ratio, months since last
# bounced cheque, unsecured credit exposure, employment length.
#
# A suspicious top-3 feature (e.g., postal code region) triggers a bias
# audit under PACT/MAS guidelines before the model ships.
#
# BUSINESS IMPACT: DBS approves ~S$12B/year in unsecured credit. A 1pp
# lift in AUC-PR is worth S$25-40M/year in avoided losses at 40-60% LGD.
# Catching a proxy-discrimination feature pre-launch avoids a ~S$5M MAS
# remediation cost.



## REFLECTION


[x] Trained XGBoost on real Singapore credit (AUC-PR={metrics['auc_pr']:.4f})
  [x] Connected every hyperparameter to the theory from 4.1
  [x] Used AUC-PR as the primary metric on imbalanced data
  [x] Ranked features by gain importance
  [x] Mapped the ranking to DBS's MAS FEAT compliance workflow

  Next: 03_lightgbm_catboost.py — the same data, two alternative
  libraries, and the decision tree for picking one.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

